In [ ]:

#%% Run to have everything 

import torch
from numpy import random
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.abspath("../tgv_pycuda-master"))
sys.path.append(os.path.abspath("../tgv_odl_pghd/denoise"))
sys.path.append(os.path.abspath(".."))




from Algo_setuptorch import Params
#from Algo_setup_tomo import Params
from data.dataset import build_train_test_data
#from data.dataset_tomo import build_train_test_data

from algorithm.unrolled_model import UnrolledFBS
from training.train import train
from NN.plots import *
from NN.run import *


from denoise_pycuda import tgv_denoise

from pghd_denoising import pdhg






device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

params = Params()

TRAIN_SEEDS = list(range(50))  
TEST_SEEDS = list(range(50,60))


size = params.size
SHAPES = [
    (1, 1, size, size),
    (1, 2, size, size),
    (1, 2, size, size),
    (1, 4, size, size),
]
N_CH = sum(s[1] for s in SHAPES)
N_CH_primal = sum(s[1] for s in SHAPES[:2])
print("ok")

train_data, test_data = build_train_test_data(
    train_seeds=TRAIN_SEEDS,
    test_seeds=TEST_SEEDS,
    params=params,
    device=device,
)


initial_state, clean, functions = test_data[0]

: 

In [ ]:

model = UnrolledFBS(
    params=params,
    shapes=SHAPES,
    n_channels=N_CH_primal,
    T=10,
    alpha=0.99,
).to(device).float()

model.load_state_dict(torch.load("model_final_10.pt", map_location="cpu"))

model.eval()
 

In [ ]:
F_vals_0, res_0= run_zero(initial_state,functions, params, SHAPES, 500, device)
F_vals,res ,history= run_learned(model,initial_state,clean,functions,T_test=500,return_all=True)
import matplotlib.pyplot as plt

img = history["x"][-1][0]

# convertir proprement
img_np = img.detach().cpu().squeeze().numpy()

plt.imsave("final_image.png", img_np, cmap="gray")

In [ ]:

# Convertir en numpy si besoin
res_0 = np.array(res_0)
res = np.array(res)

# Axe des itérations 
iters = np.arange(1, len(res_0) + 1)

plt.figure(figsize=(6,4))

# Courbes principales

plt.loglog(res_0[1:], label='Zero deviations', linewidth=2)
plt.loglog(res[1:], label='Learned deviations', linewidth=2)
# Références théoriques
plt.loglog(iters, res_0[0]/iters, '--', label=r'$O(1/t)$')
plt.loglog(iters, res_0[0]/iters**2, '--', label=r'$O(1/t^2)$')

plt.xlabel('Iterations')
plt.ylabel('Residual')
plt.title('Convergence (log-log)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# convertir proprement
F_vals_0 = np.array([
    f.detach().cpu().item() if torch.is_tensor(f) else f
    for f in F_vals_0
])

F_vals = np.array([
    f.detach().cpu().item() if torch.is_tensor(f) else f
    for f in F_vals
])

# approx de F*
F_star = min(F_vals_0.min(), F_vals.min())

# gaps
gap_0 = F_vals_0 - F_star
gap   = F_vals   - F_star

# éviter log(0)
gap_0 = np.maximum(gap_0, 1e-16)
gap   = np.maximum(gap,   1e-16)

gap_0 = gap_0[:490]
gap   =gap[:490]


iters = np.arange(1, len(gap_0) + 1)

plt.figure(figsize=(6,4))

plt.loglog(iters, gap_0, label='Zero deviations', linewidth=2)
plt.loglog(iters, gap,   label='Learned deviations', linewidth=2)

# références
plt.loglog(iters, gap_0[0]/iters, '--', label=r'$O(1/t)$')
plt.loglog(iters, gap_0[0]/iters**2, '--', label=r'$O(1/t^2)$')

plt.xlabel('Iterations')
plt.ylabel(r'$F(x_n) - F^*$')
plt.title('Convergence (log-log)')
plt.legend()
plt.grid(True, which="both", alpha=0.3)

plt.show()

In [ ]:



x, F_vals = pdhg()

F_vals_learned,res_learned = run_learned(model,initial_state,clean,functions,T_test=500)

gaps = np.array(F_vals) - min(F_vals)
gaps_learned = np.array(F_vals_learned) - min(F_vals_learned)

gaps=gaps[:1000]
gaps_learned=gaps_learned[:1000]

In [ ]:

gaps = gaps[gaps > 1e-12]
gaps_learned = gaps_learned[gaps_learned > 1e-12]
gaps /= gaps[0]
gaps_learned /= gaps_learned[0]
N = min(len(gaps), len(gaps_learned))
t = np.arange(1, N + 1)

plt.loglog(gaps[:N], label="PyCUDA TGV")
plt.loglog(gaps_learned[:N], label="My version TGV")




ref1 = 1/ t         # O(1/t)
ref2 = 1 / (t**2)    # O(1/t^2)

plt.loglog(t, ref1, '--', label=r'$O(1/t)$')
plt.loglog(t, ref2, '--', label=r'$O(1/t^2)$') 

plt.xlabel("Iterations")
plt.ylabel("Residuals")
plt.legend()
plt.minorticks_on()  # active les sous-graduations
plt.grid(which='major', linestyle='-', linewidth=0.8)
plt.grid(which='minor', linestyle='--', linewidth=0.4, alpha=0.7)

plt.savefig("energy.pdf")
plt.show()  